authors: id,name

categories: id, code, name

ingestion_runs: id, run_date, category_code, papers_fetched, papers_new, status, error_message,started_at, finished_at

paper_authors: author_id (key:authors[id]) paper_id (key:papers[id])

paper_categories: paper_id (key:papers[id]), category_id (key:categories[id]), is_primary

paper_embedings: paper_id (key:papers[id]), embedding (vector), model_used, created_at

paper_enrichments: id, paper_id (key:papers[id]), summary, methods, datasets, custom_topics, model_used, prompt_version, enriched_at

papers: id, arxiv_id, title, abstract, published_at, updated_at, pdf_url, ingested_at




In [58]:
import urllib.parse
import requests
import time  # Added for rate limiting
from lxml import etree
base_url = "http://export.arxiv.org/api/query?{parameters}"
query_groups = {
    "nlp_and_core_ai": {"categories": ["cs.CL", "cs.AI"]},
    "ml_and_foundations": {"categories": ["cs.LG", "stat.ML"]},
    "multimodal_and_vision": {"categories": ["cs.CV"]},
    "safety_and_agents": {"categories": ["cs.CR", "cs.RO"]}
}

all_papers = []

for group_name, group_data in query_groups.items():
    print(f"Fetching daily update for: {group_name}")
    cat_string = " OR ".join([f"cat:{cat}" for cat in group_data["categories"]])
    
    parameters = urllib.parse.urlencode({
        "search_query": f"({cat_string}) AND all:LLM",
        "max_results": 100,  # Top 100 newest is plenty for a single day's volume
        "sortBy": "submittedDate",
        "sortOrder": "descending"
    })
    
    response = requests.get(base_url.format(parameters=parameters))
    root = etree.fromstring(response.content)
    namespaces = {'atom': 'http://www.w3.org/2005/Atom', 'arxiv': 'http://arxiv.org/schemas/atom'}

    entries = root.findall('atom:entry', namespaces)
    for entry in entries:
        arxiv_id = entry.find('atom:id', namespaces).text.split('/')[-1]
        title = entry.find('atom:title', namespaces).text.strip()
        abstract = entry.find('atom:summary', namespaces).text.strip()
        
        # Format authors list into a comma-separated string for easier database insertion
        authors = ", ".join([author.text for author in entry.findall('atom:author/atom:name', namespaces)])
        
        cat_node = entry.find('arxiv:primary_category', namespaces)
        primary_cat = cat_node.get('term') if cat_node is not None else None
        
        all_categories = ", ".join([node.get('term') for node in entry.findall('atom:category', namespaces)])
        
        all_papers.append((arxiv_id, title, authors, abstract, primary_cat, all_categories, group_name))
    
    time.sleep(3) # Respect arXiv rate limits

print(f"Daily pull complete. Extracted {len(all_papers)} total papers.")
# return all_papers # This automatically pushes to XCom for the next task

Fetching daily update for: nlp_and_core_ai
Fetching daily update for: ml_and_foundations
Fetching daily update for: multimodal_and_vision
Fetching daily update for: safety_and_agents
Daily pull complete. Extracted 400 total papers.


Primary Category: cs.CL
All Categories: cs.CL
('YouZhi: Towards High-Concurrency Financial LLMs via Adaptive GQA-to-MLA Transition', 'cs.CL', 'cs.CL')
